# Ultimate Multi-Pair Sniper — Google Colab Training

Train per-symbol XGBoost sniper models (EURUSD, USDJPY, XAUUSD, BTCUSD) with MTF features and triple-barrier R:R 1:3 labels.

**Setup:** Upload `data/raw/{symbol}_{h1,m15,m5,m1}.csv` and the `src/` package (or open this repo in Colab).

Weekend symbol: train `BTCUSD` separately (ATR SL mult 1.5).

In [ ]:
# Install deps for Colab (avoid pandas-ta — it breaks numpy/sklearn on Colab)
# After this cell: Runtime → Restart session, then continue from the next cells.
!pip -q install "numpy==2.0.2" "scikit-learn==1.5.2" xgboost joblib pyyaml pandas

import sys
from pathlib import Path

ROOT = Path('/content') if Path('/content/src').exists() else Path('.').resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT / 'iRich' / 'src').exists():
    ROOT = ROOT / 'iRich'
    sys.path.insert(0, str(ROOT))
print('ROOT', ROOT)

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from xgboost import XGBClassifier

from src.labels import triple_barrier_labels
from src.model_runner import build_training_frame

SYMBOLS = ['EURUSD', 'USDJPY', 'XAUUSD', 'BTCUSD']
RAW = ROOT / 'data' / 'raw'
MODELS = ROOT / 'models'
MODELS.mkdir(parents=True, exist_ok=True)

LABEL_CFG = dict(atr_sl=1.0, atr_tp=3.0, horizon=45)
LABEL_ATR_SL_BY_SYMBOL = {'BTCUSD': 1.5}
MIN_PROB = 0.75

In [ ]:
def load_tf(symbol, tf):
    path = RAW / f'{symbol.lower()}_{tf}.csv'
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    return df


def train_one(symbol, use_cuda=True):
    df_h1 = load_tf(symbol, 'h1')
    df_m15 = load_tf(symbol, 'm15')
    df_m5 = load_tf(symbol, 'm5')
    df_m1 = load_tf(symbol, 'm1')

    frame, feature_cols = build_training_frame(df_h1, df_m15, df_m5, df_m1)
    frame = frame.copy()
    frame['atr_14'] = frame['m1_atr_14']
    atr_sl = LABEL_ATR_SL_BY_SYMBOL.get(symbol, LABEL_CFG['atr_sl'])
    frame['target'] = triple_barrier_labels(
        frame,
        atr_col='atr_14',
        atr_sl=atr_sl,
        atr_tp=LABEL_CFG['atr_tp'],
        horizon=LABEL_CFG['horizon'],
    )

    data = frame.dropna(subset=feature_cols + ['target'])
    X = data[feature_cols]
    y = data['target'].astype(int)
    print(symbol, 'rows', len(data), 'class balance', y.value_counts().to_dict())

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    classes, counts = np.unique(y_train, return_counts=True)
    total = counts.sum()
    weights = {int(c): float(total / (len(classes) * cnt)) for c, cnt in zip(classes, counts)}
    sample_w = y_train.map(weights)

    kwargs = dict(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
    )
    if use_cuda:
        try:
            model = XGBClassifier(**kwargs, device='cuda', tree_method='hist')
            model.fit(X_train, y_train, sample_weight=sample_w)
        except Exception as e:
            print('CUDA failed, fallback CPU:', e)
            model = XGBClassifier(**kwargs, tree_method='hist')
            model.fit(X_train, y_train, sample_weight=sample_w)
    else:
        model = XGBClassifier(**kwargs, tree_method='hist')
        model.fit(X_train, y_train, sample_weight=sample_w)

    pred = model.predict(X_test)
    print(classification_report(y_test, pred, digits=4))

    proba = model.predict_proba(X_test)
    # map to class columns
    class_index = {int(c): i for i, c in enumerate(model.classes_)}
    sniper_pred = []
    sniper_true = []
    for i in range(len(X_test)):
        p_buy = proba[i, class_index.get(1, 1)] if 1 in class_index else 0.0
        p_sell = proba[i, class_index.get(2, 2)] if 2 in class_index else 0.0
        if p_buy >= MIN_PROB or p_sell >= MIN_PROB:
            sniper_pred.append(1 if p_buy >= p_sell else 2)
            sniper_true.append(int(y_test.iloc[i]))
    if sniper_true:
        print(f'Sniper threshold {MIN_PROB} n={len(sniper_true)}')
        print(classification_report(sniper_true, sniper_pred, digits=4))

    # Feature importances
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print(imp.head(20))

    out_path = MODELS / f'{symbol.lower()}_sniper.pkl'
    joblib.dump({
        'model': model,
        'feature_cols': feature_cols,
        'meta': {'symbol': symbol, 'labels': LABEL_CFG, 'min_probability': MIN_PROB},
    }, out_path)
    print('Saved', out_path)
    return out_path

In [ ]:
for sym in SYMBOLS:
    try:
        train_one(sym, use_cuda=True)
    except FileNotFoundError as e:
        print('Missing data for', sym, e)
    except Exception as e:
        print('ERROR', sym, e)

print('Done. Download models/*_sniper.pkl to your VPS models/ folder.')